In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
import os

import numpy as np  # linear algebra
import pandas as pd  # data processing, CSV file I/O (e.g. pd.read_csv)

for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/titanic/train.csv
/kaggle/input/competitions/titanic/test.csv
/kaggle/input/competitions/titanic/gender_submission.csv


In [2]:
from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder

In [3]:
DIRPATH_INPUT = Path("/kaggle/input")
DIRPATH_WORKING = Path("/kaggle/working")
DIRPATH_TEMP = Path("/kaggle/temp")

DIRPATH_TEMP.mkdir(exist_ok=True)
dirpath_dataset = DIRPATH_INPUT / "competitions/titanic"

In [4]:
KEY_Y = "Survived"

df_train = pd.read_csv(dirpath_dataset / "train.csv", index_col=0)
X_train = df_train.drop(KEY_Y, axis=1)
y_train = df_train.loc[:, KEY_Y]
display(X_train.head(), y_train.to_frame().head())

,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,
1,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
2,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
3,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
4,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
5,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


,Survived
PassengerId,
1,0
2,1
3,1
4,1
5,0


In [5]:
X_test = pd.read_csv(dirpath_dataset / "test.csv", index_col=0)
X_test.head()

,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
PassengerId,,,,,,,,,,
892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


Embarked: 乗船地

In [6]:
X_train.isna().sum()

Pclass        0
Name          0
Sex           0
Age         177
SibSp         0
Parch         0
Ticket        0
Fare          0
Cabin       687
Embarked      2
dtype: int64

In [7]:
X_test.isna().sum()

Pclass        0
Name          0
Sex           0
Age          86
SibSp         0
Parch         0
Ticket        0
Fare          1
Cabin       327
Embarked      0
dtype: int64

In [8]:
def extract_title(X) -> np.ndarray:
    X = np.asarray(X)
    titles = ("Mr.", "Mrs.", "Miss")

    return np.array(
        [[title in name for title in titles] for name in X[:, 0]],
        dtype=int,
    )


title_transformer = FunctionTransformer(extract_title)

In [9]:
preprocessor = ColumnTransformer(
    [
        ("cat", OneHotEncoder(), ["Sex", "Embarked", "Pclass"]),
        ("num", SimpleImputer(), ["Age", "Fare"]),
        ("title", title_transformer, ["Name"]),
    ]
)

estimator = make_pipeline(preprocessor, RandomForestClassifier(random_state=0))
estimator.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['Sex', 'Embarked',
                                                   'Pclass']),
                                                 ('num', SimpleImputer(),
                                                  ['Age', 'Fare']),
                                                 ('title',
                                                  FunctionTransformer(func=<function extract_title at 0x7e189bcab1a0>),
                                                  ['Name'])])),
                ('randomforestclassifier',
                 RandomForestClassifier(random_state=0))])

In [10]:
y_pred_on_train = estimator.predict(X_train)
accuracy_score(y_train, y_pred_on_train)

0.9820426487093153

In [11]:
y_pred_on_test = estimator.predict(X_test)

In [12]:
sr_submit = pd.Series(y_pred_on_test, index=X_test.index, name=y_train.name)
sr_submit.to_frame().head()

,Survived
PassengerId,
892,0
893,0
894,0
895,1
896,1


In [13]:
sr_submit.to_csv(DIRPATH_WORKING / "submission.csv")